# 일단 ai 헬멧 감지 

In [1]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 주피터 노트북용 비동기 패치 (에러 방지)
nest_asyncio.apply()

# 오라클 클라이언트 초기화
try:
    oracledb.init_oracle_client()
except Exception as e:
    pass

# FastAPI 앱 생성
app = FastAPI()

# YOLO 모델 로딩
model_path = "data/best (sDUDU).pt"

model = None
try:
    if os.path.exists(model_path):
        model = YOLO(model_path)
        print(f"ai 모델 로딩 성공({model+path})")
        print(f"감지 가능목록(names): {model.names}")
    else:
        print(f"파일이 없음:{os.path.abspath(model_path)}")
except Exception as e:
    print(f"모델 로딩 중 에러: {e}")

# db 연결 함수
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"db접속 에러 : {e}")
        return None

# 메인 기능 : 자바가 요청 -> 카메라 켜고 감지 -> db 저장
@app.get("/helmet-check")
def helmet_check(kickboard_id: str):
    print(f"요청도착 킥보드id : {kickboard_id}")

    # ai 감지 로직
    helmet_status = "미착용"
    is_helmet_detected = False
    detected_objects = []

    if model:
        # 동영상 파일 경로
        video_path = "data/야간1(헬멧o).mp4"
        print(f"동영상({video_path})을 분석합니다.")
        
        if os.path.exists(video_path):
            # stream=True : 한번에 처리 x, 프레임별로 처리
            # max_det=1 : 한 프레임당 1개만 감지
            # vid_stride=30 : 모든 프레임x 30프레임마다 1번씩 검사
            results = model.predict(source=video_path, save=True, conf=0.5, vid_stride=30)

            # 결과 분석
            for result in results:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    class_name = model.names[cls_id]

                    # 중복 제거해 리스트에 담기
                    if class_name not in detected_objects:
                        detected_objects.append(class_name)
                        print(f"ai가본것:{class_name}")

                    # 헬멧 감지 여부 체크
                    if 'helmet' in class_name.lower():
                        is_helmet_detected = True
                        helmet_status = "착용"
                        # 헬멧을 찾으면 더 검사 X 
                        break
                if is_helmet_detected: break
        else:
            print(f"영상 파일이 없음:{video_path}")
    else:
        print(f"모델이 없음")

모델 로딩 중 에러: name 'path' is not defined


In [2]:
import os  # ★ 이 친구가 없으면 path 관련 에러가 납니다!
from ultralytics import YOLO

# 경로 설정 (아까 만든 data 폴더)
model_path = "data/best.pt"

# 디버깅용: 현재 폴더 위치와 파일이 진짜 있는지 확인
print(f"📂 현재 작업 위치: {os.getcwd()}")

if os.path.exists(model_path):  # ★ 여기에 os.path 라고 정확히 써야 합니다!
    try:
        model = YOLO(model_path)
        print(f"✅ 모델 로딩 성공! ({model_path})")
        print(f"📋 감지 목록: {model.names}")
    except Exception as e:
        print(f"💥 모델 파일은 있는데 로딩 실패: {e}")
else:
    # 파일이 없을 때 절대경로를 보여줌 (찾기 쉽게)
    print(f"💥 파일을 못 찾겠어요! 여기 있는지 확인해보세요: {os.path.abspath(model_path)}")
    model = None

📂 현재 작업 위치: C:\Users\smhrd\GitHub\RealDuDu\jupiter_python
💥 파일을 못 찾겠어요! 여기 있는지 확인해보세요: C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\data\best.pt


In [3]:
import os

print("📂 'data' 폴더 안에 있는 실제 파일 목록:")
print("-" * 30)

try:
    files = os.listdir("data") # data 폴더 안을 들여다봅니다.
    for f in files:
        print(f"👉 발견된 파일: {f}")
        
    if not files:
        print("텅 비어있는데요? 😅")
        
except Exception as e:
    print(f"💥 에러! 'data'라는 폴더 자체가 없는 것 같아요. (현재 위치에 있는 폴더들: {os.listdir('.')})")

📂 'data' 폴더 안에 있는 실제 파일 목록:
------------------------------
👉 발견된 파일: 299823_tiny.mp4
👉 발견된 파일: 39183-421020269_tiny.mp4
👉 발견된 파일: best (sDUDU).pt
👉 발견된 파일: minha.mp4
👉 발견된 파일: minha2.mp4
👉 발견된 파일: results.csv
👉 발견된 파일: results.png
👉 발견된 파일: test1.mp4
👉 발견된 파일: testvideo.mp4
👉 발견된 파일: testvideo1.mp4
👉 발견된 파일: testvideo_demo.mp4
👉 발견된 파일: 야간1(노헬멧).mp4
👉 발견된 파일: 야간1(헬멧o).mp4
👉 발견된 파일: 야간2(헬멧o).mp4


In [4]:
import os  
from ultralytics import YOLO

# 모델 경로(위치)
model_path = "data/best (sDUDU).pt"

print(f"📂 현재 위치: {os.getcwd()}")

# [수정] 그냥 path.exists가 아니라 'os.path.exists'가 맞습니다!
if os.path.exists(model_path):
    try:
        model = YOLO(model_path)
        print(f"✅ 모델 로딩 성공! ({model_path})")
        print(f"📋 클래스 목록(names): {model.names}") 
    except Exception as e:
        print(f"💥 파일은 있는데 로딩 실패: {e}")
else:
    # [수정] 여기도 path.abspath가 아니라 'os.path.abspath'
    print(f"💥 파일을 못 찾겠어요: {os.path.abspath(model_path)}")
    model = None

📂 현재 위치: C:\Users\smhrd\GitHub\RealDuDu\jupiter_python
✅ 모델 로딩 성공! (data/best (sDUDU).pt)
📋 클래스 목록(names): {0: 'no_helmet', 1: 'helmet'}


In [7]:
# ==========================================
# [1] 필요한 도구(라이브러리)들을 가져오는 구역
# ==========================================
import uvicorn              # 서버를 실행시켜주는 도구 (웹 서버)
import nest_asyncio         # 주피터 노트북에서 서버가 에러 없이 돌게 해주는 패치
from fastapi import FastAPI # 웹 페이지 요청을 받아주는 핵심 도구
import oracledb             # 오라클 DB와 대화하기 위한 전화기
import datetime             # 날짜와 시간을 다루는 도구
import os                   # 파일 경로(폴더 위치)를 확인하는 도구
from ultralytics import YOLO # AI(YOLO) 모델을 사용하는 도구

# 주피터 노트북은 원래 비동기 작업(서버 돌리기 등)을 막아놓는데, 
# 이걸 풀어주는 코드입니다. (이거 없으면 에러남!)
nest_asyncio.apply()

# ==========================================
# [2] 기본 설정 (오라클 & 웹 서버)
# ==========================================

# 오라클 클라이언트(접속 프로그램) 초기화
# 가끔 설치 안 된 PC에서 에러가 나서 try-except로 감싸둠
try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

# FastAPI 앱(서버 본체) 생성
app = FastAPI()

# ==========================================
# [3] AI 모델 로딩 (가장 중요한 뇌 부분)
# ==========================================

# 우리가 사용할 학습된 모델 파일 경로
# 주의: 파일명 띄어쓰기, 괄호까지 정확해야 함!
model_path = "data/best (sDUDU).pt" 
model = None # 일단 빈 변수로 시작

# 파일이 진짜 있는지 확인하고 로딩
if os.path.exists(model_path):
    # 모델 파일을 읽어와서 메모리에 올림
    model = YOLO(model_path)
    print(f"✅ AI 모델 로딩 성공! ({model_path})")
    print(f"📋 이 모델이 아는 것들: {model.names}") 
    # {0: 'no_helmet', 1: 'helmet'} 확인됨
else:
    print(f"💥 모델 파일이 없어요! 경로 확인필요: {model_path}")

# ==========================================
# [4] DB 연결 함수 (전화기 들기)
# ==========================================
def get_db_connection():
    try:
        # DB 접속 정보 입력 (주소, 포트, 서비스이름)
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        
        # 실제 연결 시도 (아이디, 비밀번호)
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn # 연결 성공하면 전화기(conn)를 반환
    except Exception as e:
        print(f"❌ DB 접속 실패: {e}")
        return None

# ==========================================
# [5] 메인 기능: 헬멧 감지 요청 처리
# ==========================================
# 자바가 "http://IP:8001/helmet-check?kickboard_id=..." 로 접속하면 이 함수가 실행됨
@app.get("/helmet-check")
def helmet_check(kickboard_id: str):
    
    # 1. 요청이 왔음을 알림
    print(f"📡 [요청 도착] 킥보드 ID: {kickboard_id}")
    
    # --- [A] AI 감지 로직 (영상 분석) ---
    helmet_status = "미착용"     # 일단 '미착용'이라고 가정 (안전빵)
    is_helmet_detected = False   # 헬멧 쓴 사람 발견했니? (아직 못 봄)
    detected_list = []           # 로그 확인용 리스트

    if model: # 모델이 정상적으로 로딩되었을 때만 실행
        
        # 분석할 테스트 영상 파일 경로
        video_path = "data/야간1(헬멧o).mp4" 
        print(f"🎬 영상 분석 시작: {video_path}")
        
        if os.path.exists(video_path):
            # predict: 예측해라! 
            # source: 영상파일, save: 결과사진 저장, conf: 50% 이상 확실할 때만
            # vid_stride=30: 속도를 위해 30프레임마다 1번씩만 검사 (매우 중요!)
            results = model.predict(source=video_path, save=True, conf=0.5, vid_stride=30, verbose=False)
            
            # 분석 결과(results)를 하나씩 까봅니다
            for result in results:
                for box in result.boxes:
                    # box.cls[0]: 감지된 물체의 번호 (0 또는 1)
                    cls_id = int(box.cls[0])           
                    # model.names[cls_id]: 번호를 이름으로 바꿈 ('helmet' 등)
                    class_name = model.names[cls_id]   
                    
                    # 로그 찍기 (중복 방지)
                    if class_name not in detected_list:
                        detected_list.append(class_name)
                        print(f"🧐 AI 발견: {class_name}")

                    # ★ 핵심 규칙 ★
                    # 모델이 알려준 번호가 1번(helmet)이면 '착용'으로 인정!
                    if cls_id == 1: 
                        is_helmet_detected = True
                        helmet_status = "착용"
                        
            print(f"📊 최종 판단 결과: {helmet_status}")
            
        else:
            print(f"💥 영상 파일이 없어서 분석을 못해요: {video_path}")
    
    # --- [B] DB 저장 로직 ---
    result_msg = "실패"
    conn = get_db_connection() # DB 접속
    
    if conn:
        try:
            cursor = conn.cursor() # 쿼리 날릴 준비
            
            # 현재 시간 구하기
            now = datetime.datetime.now()
            # RIDE_ID 만들기 (예: RIDE_20250115143000)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            # 점수 계산: 헬멧 썼으면 10점, 안 썼으면 -5점 벌점
            score_cg = 10 if is_helmet_detected else -5
            # 미착용 횟수: 썼으면 0, 안 썼으면 1
            no_helmet_cnt = 0 if is_helmet_detected else 1
            
            # DB에 넣을 데이터 포장하기 (순서 중요!)
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,   
                "USER010",      # 회원 ID (실제 DB에 있는 사람)
                kickboard_id,  # 자바에서 받아온 킥보드 ID (DD010)
                now, now,      # 시작시간, 종료시간
                12.5,          # 주행시간 (임시값)
                no_helmet_cnt, # 미착용 횟수
                score_cg,      # 점수
                "P"            # 운행상태 (Parking 등 규칙에 맞는 값)
            )
            
            # 쿼리 실행 및 저장(commit)
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 점수: {score_cg})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러 발생: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close() # 전화 끊기 (필수)
            
    # 자바(안드로이드)에게 결과를 JSON 형태로 돌려줌
    return {
        "result": result_msg,
        "kickboard_id": kickboard_id,
        "helmet_check": helmet_status,
        "detected": detected_list
    }

# ==========================================
# [6] 서버 실행 구역
# ==========================================
if __name__ == "__main__":
    # 포트 8001번으로 서버를 엽니다.
    # host="0.0.0.0"은 외부(팀원 컴퓨터) 접속을 허용한다는 뜻
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 로딩 성공! (data/best (sDUDU).pt)
📋 이 모델이 아는 것들: {0: 'no_helmet', 1: 'helmet'}


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드 ID: DD010
🎬 영상 분석 시작: data/야간1(헬멧o).mp4
WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict
🧐 AI 발견: helmet
🧐 AI 발견: no_helmet
📊 최종 판단 결과: 착용
💾 DB 저장 완료! (ID: RIDE_20260115160113, 점수: 10)
INFO:     192.168.219.166:58505 - "GET /helmet-check?kickboard_id=DD010 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]


In [13]:
import os

print("📂 'data' 폴더 파일 목록:")
print("-" * 30)
# data 폴더 안의 내용을 보여줘!
files = os.listdir("data") 
for f in files:
    print(f)

📂 'data' 폴더 파일 목록:
------------------------------
299823_tiny.mp4
39183-421020269_tiny.mp4
best (sDUDU).pt
minha.mp4
minha2.mp4
results.csv
results.png
test1.mp4
testvideo.mp4
testvideo1.mp4
testvideo_demo.mp4
야간1(노헬멧)편집.mp4
야간1(헬멧o)편집.mp4
야간2(헬멧o)편집.mp4
주간1(노헬멧).mp4
주간1(헬멧o).mp4


# 점수 계산식 적용버전(비율 90%)

In [14]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 1. 설정
nest_asyncio.apply()

try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# [핵심 변경] user_id를 파라미터로 추가했습니다!
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] AI 감지 로직
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # ★ 영상 경로 확인 필수 (아까 성공한 파일명으로 해둠)
    video_path = "data/주간1(헬멧o).mp4" 

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    break
            
            if found_something:
                total_seconds += 1
                
        # [B] 점수 계산 로직 (경고음 삭제됨)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (90% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"헬멧 착용 O (+{plus_score:.2f}점")

            # (2) 미착용 모드 (90% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 30:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (30초마다)
                    extra = (no_helmet_count - 30) // 30
                    if extra > 0:
                        minus_score += (extra * 1)
                final_score -= minus_score
                print(f"미착용 확정!(-{minus_score}점")
            else:
                helmet_status = "판독애매"
                print("비율이 애매해서 점수 변동 없음")
    else:
        print("⚠️ 감지된 시간이 너무 짧아서 계산 불가")

    # [C] DB 저장 로직 (완전 자동화)
    result_msg = "실패"
    conn = get_db_connection()
    
    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성 (시간 기반)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    # 자동 생성된 ID
                user_id,        # ★ 자바에서 받아온 진짜 유저 ID
                kickboard_id,   # ★ 자바에서 받아온 진짜 킥보드 ID
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 유저: {user_id}, 점수: {final_score:.2f})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id if 'new_ride_id' in locals() else "생성실패",
        "score": round(final_score, 2),
        "status": helmet_status
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD010 / 사용자: USER001
🎬 분석 시작: data/주간1(헬멧o).mp4
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict5
헬멧 착용 O (+0.02점
💾 DB 저장 완료! (ID: RIDE_20260115172143, 유저: USER001, 점수: 0.02)
INFO:     192.168.219.166:63475 - "GET /helmet-check?kickboard_id=DD010&user_id=USER001 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]


In [17]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 1. 설정
nest_asyncio.apply()

try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# [핵심 변경] user_id를 파라미터로 추가했습니다!
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] AI 감지 로직
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # ★ 영상 경로 확인 필수 (아까 성공한 파일명으로 해둠)
    video_path = "data/주간1(노헬멧).mp4" 

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    break
            
            if found_something:
                total_seconds += 1
                
        # [B] 점수 계산 로직 (경고음 삭제됨)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (90% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"헬멧 착용 O (+{plus_score:.2f}점")

            # (2) 미착용 모드 (90% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 5:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (30초마다)
                    extra = (no_helmet_count - 5) // 5
                    if extra > 0:
                        minus_score += (extra * 1)
                final_score -= minus_score
                print(f"미착용 확정!(-{minus_score}점")
            else:
                helmet_status = "판독애매"
                print("비율이 애매(반반)해서 점수 변동 없음")
    else:
        print("⚠️ 감지된 시간이 너무 짧아서 계산 불가")

    # [C] DB 저장 로직 (완전 자동화)
    result_msg = "실패"
    conn = get_db_connection()
    
    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성 (시간 기반)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    # 자동 생성된 ID
                user_id,        # ★ 자바에서 받아온 진짜 유저 ID
                kickboard_id,   # ★ 자바에서 받아온 진짜 킥보드 ID
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 유저: {user_id}, 점수: {final_score:.2f})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id if 'new_ride_id' in locals() else "생성실패",
        "score": round(final_score, 2),
        "status": helmet_status
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD010 / 사용자: USER001
🎬 분석 시작: data/주간1(노헬멧).mp4
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict7
미착용 확정!(-3점
💾 DB 저장 완료! (ID: RIDE_20260115172700, 유저: USER001, 점수: -3.00)
INFO:     192.168.219.166:55410 - "GET /helmet-check?kickboard_id=DD010&user_id=USER001 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]


In [18]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 1. 설정
nest_asyncio.apply()

try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# [핵심 변경] user_id를 파라미터로 추가했습니다!
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] AI 감지 로직
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # ★ 영상 경로 확인 필수 (아까 성공한 파일명으로 해둠)
    video_path = "data/야간1(헬멧o)편집.mp4" 

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    break
            
            if found_something:
                total_seconds += 1
                
        # [B] 점수 계산 로직 (경고음 삭제됨)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (90% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"헬멧 착용 O (+{plus_score:.2f}점")

            # (2) 미착용 모드 (90% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 5:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (30초마다)
                    extra = (no_helmet_count - 5) // 5
                    if extra > 0:
                        minus_score += (extra * 1)
                final_score -= minus_score
                print(f"미착용 확정!(-{minus_score}점")
            else:
                helmet_status = "판독애매"
                print("비율이 애매(반반)해서 점수 변동 없음")
    else:
        print("⚠️ 감지된 시간이 너무 짧아서 계산 불가")

    # [C] DB 저장 로직 (완전 자동화)
    result_msg = "실패"
    conn = get_db_connection()
    
    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성 (시간 기반)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    # 자동 생성된 ID
                user_id,        # ★ 자바에서 받아온 진짜 유저 ID
                kickboard_id,   # ★ 자바에서 받아온 진짜 킥보드 ID
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 유저: {user_id}, 점수: {final_score:.2f})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id if 'new_ride_id' in locals() else "생성실패",
        "score": round(final_score, 2),
        "status": helmet_status
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD010 / 사용자: USER001
🎬 분석 시작: data/야간1(헬멧o)편집.mp4
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict8
헬멧 착용 O (+0.02점
💾 DB 저장 완료! (ID: RIDE_20260115172836, 유저: USER001, 점수: 0.02)
INFO:     192.168.219.166:55415 - "GET /helmet-check?kickboard_id=DD010&user_id=USER001 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]


In [19]:
import uvicorn
import nest_asyncio
from fastapi import FastAPI
import oracledb
import datetime
import os
from ultralytics import YOLO

# 1. 설정
nest_asyncio.apply()

try:
    oracledb.init_oracle_client()
except Exception as e:
    pass 

app = FastAPI()

# 2. AI 모델 로딩
model_path = "data/best (sDUDU).pt"
model = None

if os.path.exists(model_path):
    model = YOLO(model_path)
    print(f"✅ AI 모델 준비 완료! ({model_path})")
else:
    print(f"💥 모델 파일 없음: {model_path}")

# 3. DB 연결
def get_db_connection():
    try:
        dsn = oracledb.makedsn("project-db-campus.smhrd.com", 1524, sid="xe")
        conn = oracledb.connect(user="campus_25IS_GA2_p2_4", password="smhrd4", dsn=dsn)
        return conn
    except Exception as e:
        print(f"❌ DB 접속 에러: {e}")
        return None

# =========================================================
# [핵심 변경] user_id를 파라미터로 추가했습니다!
# =========================================================
@app.get("/helmet-check")
def helmet_check(kickboard_id: str, user_id: str):
    print(f"📡 [요청 도착] 킥보드: {kickboard_id} / 사용자: {user_id}")
    
    # [A] AI 감지 로직
    helmet_count = 0        
    no_helmet_count = 0     
    total_seconds = 0       
    helmet_status = "판독불가" 
    
    # ★ 영상 경로 확인 필수 (아까 성공한 파일명으로 해둠)
    video_path = "data/야간1(노헬멧)편집.mp4" 

    if model and os.path.exists(video_path):
        print(f"🎬 분석 시작: {video_path}")
        results = model.predict(source=video_path, save=True, conf=0.25, vid_stride=30, stream=True, verbose=False)
        
        for result in results:
            found_something = False
            for box in result.boxes:
                cls_id = int(box.cls[0]) # 0:no_helmet, 1:helmet
                
                if cls_id == 1: # 헬멧 씀
                    helmet_count += 1
                    found_something = True
                    break 
                elif cls_id == 0: # 헬멧 안 씀
                    no_helmet_count += 1
                    found_something = True
                    break
            
            if found_something:
                total_seconds += 1
                
        # [B] 점수 계산 로직 (경고음 삭제됨)
        final_score = 0
        
        if total_seconds > 0:
            helmet_ratio = helmet_count / total_seconds
            no_helmet_ratio = no_helmet_count / total_seconds
            
            # (1) 헬멧 착용 모드 (90% 이상) -> 가점
            if helmet_ratio >= 0.6:
                helmet_status = "착용(우수)"
                ride_minutes = total_seconds / 60
                plus_score = ride_minutes * 0.1
                final_score += plus_score
                print(f"헬멧 착용 O (+{plus_score:.2f}점")

            # (2) 미착용 모드 (90% 이상) -> 감점
            elif no_helmet_ratio >= 0.6:
                helmet_status = "미착용(위험)"
                minus_score = 0
                
                if no_helmet_count >= 3:
                    minus_score += 2 # 기본 감점
                    # 추가 감점 (30초마다)
                    extra = (no_helmet_count - 3) // 3
                    if extra > 0:
                        minus_score += (extra * 1)
                final_score -= minus_score
                print(f"미착용 확정!(-{minus_score}점")
            else:
                helmet_status = "판독애매"
                print("비율이 애매(반반)해서 점수 변동 없음")
    else:
        print("⚠️ 감지된 시간이 너무 짧아서 계산 불가")

    # [C] DB 저장 로직 (완전 자동화)
    result_msg = "실패"
    conn = get_db_connection()
    
    if conn:
        try:
            cursor = conn.cursor()
            now = datetime.datetime.now()
            
            # 1. RIDE_ID 자동 생성 (시간 기반)
            new_ride_id = "RIDE_" + now.strftime("%Y%m%d%H%M%S")
            
            sql = """
                INSERT INTO TB_RIDE (
                    RIDE_ID, USER_ID, KICKBOARD_ID, START_DT, END_DT, 
                    HELMET_TM, NOHELMET_CNT, SCORE_CG, RIDE_ST
                ) 
                VALUES (:1, :2, :3, :4, :5, :6, :7, :8, :9)
            """
            
            data = (
                new_ride_id,    # 자동 생성된 ID
                user_id,        # ★ 자바에서 받아온 진짜 유저 ID
                kickboard_id,   # ★ 자바에서 받아온 진짜 킥보드 ID
                now, now,
                total_seconds,  
                no_helmet_count, 
                round(final_score, 2), 
                "P"             
            )
            
            cursor.execute(sql, data)
            conn.commit()
            print(f"💾 DB 저장 완료! (ID: {new_ride_id}, 유저: {user_id}, 점수: {final_score:.2f})")
            result_msg = "성공"
            
        except Exception as e:
            print(f"💥 DB 에러: {e}")
            result_msg = f"DB에러: {e}"
        finally:
            conn.close()
            
    return {
        "result": result_msg,
        "ride_id": new_ride_id if 'new_ride_id' in locals() else "생성실패",
        "score": round(final_score, 2),
        "status": helmet_status
    }

# 서버 실행
if __name__ == "__main__":
    config = uvicorn.Config(app=app, host="0.0.0.0", port=8001)
    server = uvicorn.Server(config)
    await server.serve()

✅ AI 모델 준비 완료! (data/best (sDUDU).pt)


INFO:     Started server process [9180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8001 (Press CTRL+C to quit)


📡 [요청 도착] 킥보드: DD010 / 사용자: USER001
🎬 분석 시작: data/야간1(노헬멧)편집.mp4
Results saved to C:\Users\smhrd\GitHub\RealDuDu\jupiter_python\runs\detect\predict9
미착용 확정!(-5점
💾 DB 저장 완료! (ID: RIDE_20260115172948, 유저: USER001, 점수: -5.00)
INFO:     192.168.219.166:55417 - "GET /helmet-check?kickboard_id=DD010&user_id=USER001 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9180]
